# Lab 19-05 · Wires: Distributed RC, Repeaters and Crosstalk

**Predict first**, then run. The handout is [README.md](README.md).

In [1]:
import math
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import wires as wr

## Part 1 · The distributed line

*Predict* where the far end of a step-driven line reaches 50%, in units of RC.

In [2]:
th = np.linspace(0.001, 2, 400)
plt.plot(th, [wr.line_step_response(x) for x in th], label="distributed")
plt.plot(th, 1 - np.exp(-th), "--", label="lumped")
plt.plot(th, 1 - np.exp(-2 * th), ":", label="Elmore pole")
plt.axhline(0.5, color="gray"); plt.xlabel("t / RC"); plt.legend(); plt.show()
print(f"distributed 50%: {wr.line_t50():.4f} RC; lumped {math.log(2):.4f}; Elmore pole {math.log(2) / 2:.4f}")

distributed 50%: 0.3787 RC; lumped 0.6931; Elmore pole 0.3466


## Part 2 · Ladders

*Predict* how many sections bring a ladder within 1% of the distributed line.

In [3]:
exact = wr.line_t50()
for n in (5, 10, 20, 50, 100):
    t = wr.ladder_t50(n, 1.0, 1.0)
    print(f"{n:3d} sections: {t:.4f} RC, {100 * (t / exact - 1):.2f}% above the distributed line")
RW, CW = 13.3e3, 198e-15
print("100-section ladder, 1 mm wire:", round(wr.ladder_t50(100, RW, CW) * 1e12, 1), "ps bare;",
      round(wr.ladder_t50(100, RW, CW, 1e3, 10e-15) * 1e12, 1), "ps with driver and load (ngspice: rcline.cir)")

  5 sections: 0.4529 RC, 19.59% above the distributed line
 10 sections: 0.4162 RC, 9.90% above the distributed line
 20 sections: 0.3976 RC, 4.98% above the distributed line
 50 sections: 0.3863 RC, 2.00% above the distributed line


100 sections: 0.3825 RC, 1.00% above the distributed line


100-section ladder, 1 mm wire: 1007.4 ps bare; 1262.8 ps with driver and load (ngspice: rcline.cir)


## Part 3 · Driver, wire and load

*Predict* Example 19-05.3(b)'s delay from the formula, and how far it is from the converged ladder.

In [4]:
est = wr.driver_wire_load_delay(1e3, RW, CW, 10e-15)
t200, t400 = wr.ladder_t50(200, RW, CW, 1e3, 10e-15), wr.ladder_t50(400, RW, CW, 1e3, 10e-15)
cont = 2 * t400 - t200
print(f"formula {est * 1e12:.1f} ps; ladders 200/400 sections {t200 * 1e12:.1f}/{t400 * 1e12:.1f} ps; extrapolated {cont * 1e12:.1f} ps; formula {100 * (1 - est / cont):.1f}% low")

formula 1234.4 ps; ladders 200/400 sections 1258.0/1255.6 ps; extrapolated 1253.2 ps; formula 1.5% low


## Part 4 · Repeaters

*Predict* the penalty for half or twice the optimal number and size of repeaters.

In [5]:
k, h, T = wr.repeater_optimum(10e3, 0.1e-15, RW, CW)
print(f"k = {k:.1f}, h = {h:.1f}, T = {T * 1e12:.1f} ps")
for f in (0.5, 2):
    print(f"k x {f}: +{100 * (wr.repeated_delay(k * f, h, 10e3, 0.1e-15, RW, CW) / T - 1):.1f}%;  h x {f}: +{100 * (wr.repeated_delay(k, h * f, 10e3, 0.1e-15, RW, CW) / T - 1):.1f}%")
ks = np.linspace(5, 120, 200)
plt.plot(ks, [wr.repeated_delay(x, h, 10e3, 0.1e-15, RW, CW) * 1e12 for x in ks]); plt.xlabel("number of segments k"); plt.ylabel("delay (ps)"); plt.show()

k = 37.9, h = 38.6, T = 123.7 ps
k x 0.5: +10.6%;  h x 0.5: +14.4%
k x 2: +10.6%;  h x 2: +14.4%


## Part 5 · Crosstalk and scaling

*Predict* the floating-victim disturbance against C_c/C_g, and the repeated delay per millimetre if R0 C0 halves.

In [6]:
ratio = np.linspace(0, 1, 11)
print([round(wr.floating_victim_noise(1.0, x, 1.0), 3) for x in ratio])
print("delay per mm with R0C0 = 1 ps:", round(T * 1e12, 1), "ps; with 0.5 ps:", round(wr.repeater_optimum(5e3, 0.1e-15, RW, CW)[2] * 1e12, 1), "ps")

[np.float64(0.0), np.float64(0.091), np.float64(0.167), np.float64(0.231), np.float64(0.286), np.float64(0.333), np.float64(0.375), np.float64(0.412), np.float64(0.444), np.float64(0.474), np.float64(0.5)]
delay per mm with R0C0 = 1 ps: 123.7 ps; with 0.5 ps: 87.5 ps


## Discussion

The questions and worked answers are in the [handout](README.md#discussion).